In [0]:
 %pip install openpyxl 

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
from pyspark.sql.functions import col, to_date, when
from pyspark.sql.types import DoubleType, IntegerType

# from pyspark.sql import SparkSession

# spark = SparkSession.builder \
#     .appName("ExcelReader") \
#     .config("spark.jars.packages", "com.crealytics:spark-excel_2.12") \
#     .getOrCreate()

Cleaning Customers.json


In [0]:
customers_df = spark.read.format("json").option("multiline", True).json("/Volumes/workspace/default/data/customers.json")
customers_clean = customers_df.dropna(subset=["customer_id"]).dropDuplicates(["customer_id"])
# display(customers_clean)


Cleaning product.xlsx


In [0]:
products_df_pandas = pd.read_excel("/Volumes/workspace/default/data/products.xlsx", 
                         sheet_name="Sheet1")
products_df = spark.createDataFrame(products_df_pandas )
# display(products_df)
products_clean = products_df.fillna({"price": 0})

display(products_clean)


product_id,product_name,category,price
PID1000,Drug_0,Antiviral,96.96
PID1001,Drug_1,Analgesic,15.41
PID1002,Drug_2,Antifungal,57.58
PID1003,Drug_3,Analgesic,54.04
PID1004,Drug_4,Analgisec,36.1
PID1005,Drug_5,Analgesic,66.87
PID1006,Drug_6,Antiviral,88.34
PID1007,Drug_7,Antiviral,0.0
PID1008,Drug_8,Vaccine,32.13
PID1009,Drug_9,Antifungal,91.62


cleaning inventory.xlsx


In [0]:
inventory_df_pandas = pd.read_excel("/Volumes/workspace/default/data/inventory.xlsx", 
                         sheet_name="Sheet1")

# print(inventory_df_pandas["stock_level"].unique())

string_to_int_map = {
    "zero": 0,
    "ten": 10,
    "twenty": 20,
    "one hundred": 100
}

inventory_df_pandas["stock_level"] = inventory_df_pandas["stock_level"].replace(string_to_int_map)

inventory_df_pandas["stock_level"] = pd.to_numeric(inventory_df_pandas["stock_level"], errors="coerce")

inventory_df = spark.createDataFrame(inventory_df_pandas)

inventory_clean = inventory_df.fillna({"stock_level": 0}).withColumn("stock_level", col("stock_level").cast(IntegerType()))\
                .withColumn("reorder_level", col("reorder_level").cast(IntegerType()))\
                .withColumn("avg_daily_sales", col("avg_daily_sales").cast(DoubleType()))\
                .dropna(subset=[
                    "warehouse_id", "product_id", "stock_level",
                    "reorder_level", "avg_daily_sales", "days_until_reorder"
                ])\
                .filter(
                    (col("stock_level") >= 0)
                    
                )
display(inventory_clean)


warehouse_id,product_id,stock_level,reorder_level,avg_daily_sales,days_until_reorder
W000_000,PID1000,29,46,17.0,1.7
W000_001,PID1001,153,66,92.0,1.7
W000_002,PID1002,78,78,60.0,1.3
W000_003,PID1003,0,67,81.0,1.5
W000_004,PID1004,173,41,100.0,1.7
W000_005,PID1005,150,60,86.0,1.7
W000_006,PID1006,51,87,32.0,1.6
W000_007,PID1007,103,26,65.0,1.6
W000_008,PID1008,140,45,81.0,1.7
W000_009,PID1009,97,74,54.0,1.8


cleaning sales folder files

In [0]:
sales_df = spark.read.csv("/Volumes/workspace/default/data/sales/*", header=True, inferSchema=True)    
sales_clean =  sales_df.withColumn("sale_date", to_date(col("sale_date"), "yyyy-MM-dd"))\
    .dropna(subset=["sale_date", "product_id", "customer_id", "quantity", "product_price","total_sale_amount"])

display(sales_clean)

sale_id,sale_date,product_id,customer_id,quantity,product_price,total_sale_amount
S00428575,2023-01-07,PID1001,C1038,6.0,61.63,369.78
S00428576,2023-01-07,PID1001,C1044,12.0,61.63,739.56
S00428577,2023-01-07,PID1003,C1015,13.0,62.34,810.42
S00428578,2023-01-07,PID1002,C1027,2.0,48.97,97.94
S00428579,2023-01-07,PID1004,C1011,1.0,19.46,19.46
S00428580,2023-01-07,PID1000,C1003,4.0,33.76,135.04
S00428581,2023-01-07,PID1004,C1042,2.0,19.46,38.92
S00428582,2023-01-07,PID1003,C1040,10.0,62.34,623.4
S00428583,2023-01-07,PID1002,C1008,4.0,48.97,195.88
S00428584,2023-01-07,PID1001,C1042,14.0,61.63,862.82


Transformation

In [0]:
sales_product_df = sales_clean.join(products_clean, on="product_id", how="inner")

# Join the previous result with customer_df on customer_id
final_df = sales_product_df.join(customers_clean, on="customer_id", how="inner")

# Show the final DataFrame after join
display(final_df)

customer_id,product_id,sale_id,sale_date,quantity,product_price,total_sale_amount,product_name,category,price,city,customer_name,region,sales_rep,type
C1038,PID1001,S00428575,2023-01-07,6.0,61.63,369.78,Drug_1,Analgesic,15.41,Pune,Jeffrey Sanchez II,N/E,Rep_1,Retail
C1044,PID1001,S00428576,2023-01-07,12.0,61.63,739.56,Drug_1,Analgesic,15.41,Los Angeles,Megan Sherman,West,Rep_1,Online
C1015,PID1003,S00428577,2023-01-07,13.0,62.34,810.42,Drug_3,Analgesic,54.04,Basel,Thomas Ortiz,South,Rep_5,Wholesale
C1027,PID1002,S00428578,2023-01-07,2.0,48.97,97.94,Drug_2,Antifungal,57.58,Phoenix,Roger Macdonald,East,Rep_6,Hospital
C1011,PID1004,S00428579,2023-01-07,1.0,19.46,19.46,Drug_4,Analgisec,36.1,Los Angeles,Tom Hester,West,Rep_3,Retail
C1003,PID1000,S00428580,2023-01-07,4.0,33.76,135.04,Drug_0,Antiviral,96.96,Delhi,Claudia Rodriguez,East,Rep_7,Retail
C1042,PID1004,S00428581,2023-01-07,2.0,19.46,38.92,Drug_4,Analgisec,36.1,Chicago,Susan Richardson,Northeast,Rep_4,Retail
C1040,PID1003,S00428582,2023-01-07,10.0,62.34,623.4,Drug_3,Analgesic,54.04,Phoenix,Natalie Perkins,Northeast,Rep_7,Online
C1008,PID1002,S00428583,2023-01-07,4.0,48.97,195.88,Drug_2,Antifungal,57.58,Delhi,Matthew Robinson,Central,Rep_6,Retail
C1042,PID1001,S00428584,2023-01-07,14.0,61.63,862.82,Drug_1,Analgesic,15.41,Chicago,Susan Richardson,Northeast,Rep_4,Retail


In [0]:
final_df.write.mode("overwrite").parquet("/Volumes/workspace/default/test")